# อ่านค่า Analog จาก PLC และบันทึก CSV

Notebook นี้อ่าน Holding Register `D0` ผ่าน Modbus RTU แปลงค่า ADC 0–4095 เป็นแรงดัน 0–10 V และบันทึกลง `plc_voltage_log.csv`

> ตรวจสอบ `COM_PORT`, `BAUDRATE` และ `SLAVE_ID` ก่อนรัน cell สุดท้าย กด **Interrupt Kernel** หรือ `Ctrl+C` เพื่อหยุดบันทึกข้อมูล

In [11]:
import csv
from datetime import datetime
from pathlib import Path
import time

import pymodbus
from pymodbus.client import ModbusSerialClient
from pymodbus.exceptions import ModbusException

print(f"PyModbus version: {pymodbus.__version__}")

PyModbus version: 3.15.0


In [12]:
# --- ตั้งค่าการเชื่อมต่อ ---
COM_PORT = "COM6"  # พอร์ตที่ตรวจพบในเครื่องนี้
BAUDRATE = 9600       # ต้องตรงกับค่า D8120 ของ PLC
SLAVE_ID = 1          # Station ID ของ PLC
REGISTER_ADDRESS = 0  # D0
SAMPLE_INTERVAL = 1.0 # วินาที
CSV_FILENAME = Path("plc_voltage_log.csv")

ADC_MAX = 4095.0
FULL_SCALE_VOLTAGE = 10.0

In [13]:
def raw_to_voltage(raw_value):
    """แปลงค่า ADC 12-bit (0–4095) เป็นแรงดัน 0–10 V"""
    return round((raw_value / ADC_MAX) * FULL_SCALE_VOLTAGE, 2)


def prepare_csv(filename=CSV_FILENAME):
    """สร้างไฟล์ CSV และ header เมื่อไฟล์ยังไม่มีข้อมูล"""
    if filename.exists() and filename.stat().st_size > 0:
        return

    with filename.open(mode="w", newline="", encoding="utf-8-sig") as file:
        csv.writer(file).writerow(["Timestamp", "Raw_AD (D0)", "Voltage_V"])


def build_client():
    """สร้าง Modbus RTU client สำหรับ PyModbus 3.15"""
    return ModbusSerialClient(
        port=COM_PORT,
        baudrate=BAUDRATE,
        bytesize=8,
        parity="N",
        stopbits=1,
        timeout=1,
    )


def read_register(client):
    """อ่าน D0 จาก Modbus device ที่กำหนด"""
    return client.read_holding_registers(
        address=REGISTER_ADDRESS,
        count=1,
        device_id=SLAVE_ID,
    )

In [14]:
def log_plc_data():
    prepare_csv()
    client = build_client()

    if not client.connect():
        client.close()
        raise ConnectionError(
            f"ไม่สามารถเปิดพอร์ต {COM_PORT} ได้ กรุณาตรวจสอบพอร์ตและการต่อสาย"
        )

    print(
        f"เชื่อมต่อ {COM_PORT} สำเร็จ กำลังอ่าน D0 ทุก {SAMPLE_INTERVAL:g} วินาที "
        "(กด Ctrl+C เพื่อหยุด)"
    )

    try:
        while True:
            try:
                result = read_register(client)

                if result is None or result.isError() or not hasattr(result, "registers"):
                    print("PLC ไม่ตอบสนอง กำลังรอสัญญาณ...")
                else:
                    raw_value = result.registers[0]
                    voltage = raw_to_voltage(raw_value)
                    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                    with CSV_FILENAME.open(
                        mode="a", newline="", encoding="utf-8-sig"
                    ) as file:
                        csv.writer(file).writerow([now_str, raw_value, voltage])

                    print(f"[{now_str}] D0: {raw_value} | แรงดัน: {voltage} V")

            except ModbusException as error:
                print(f"ไม่ได้รับสัญญาณตอบกลับจาก PLC: {error}")

            time.sleep(SAMPLE_INTERVAL)

    except KeyboardInterrupt:
        print("\nหยุดการบันทึกข้อมูลเรียบร้อย")
    finally:
        client.close()

In [15]:
# รัน cell นี้เพื่อเริ่มอ่านข้อมูลจาก PLC
log_plc_data()

เชื่อมต่อ COM6 สำเร็จ กำลังอ่าน D0 ทุก 1 วินาที (กด Ctrl+C เพื่อหยุด)
[2026-09-19 11:50:48] D0: 0 | แรงดัน: 0.0 V
[2026-09-19 11:50:49] D0: 0 | แรงดัน: 0.0 V
[2026-09-19 11:50:50] D0: 0 | แรงดัน: 0.0 V
[2026-09-19 11:50:52] D0: 0 | แรงดัน: 0.0 V

หยุดการบันทึกข้อมูลเรียบร้อย
